<a href="https://colab.research.google.com/github/djdhawal/bayesian-failure-prediction-on-cluster-traces/blob/main/notebooks/validation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# load the drive helper and mount
from google.colab import drive

# this will prompt for authorization. Please Accept the auth request
drive.mount('/content/drive')

path = "/content/drive/MyDrive/Bayesian Final Project/DATA"
## now you should be able to see the drive folder accessible by collab

Mounted at /content/drive


In [2]:
pip install numpyro

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 384.1/384.1 kB 22.2 MB/s eta 0:00:00


In [3]:
import pandas as pd
import numpy as np
import jax.numpy as jnp
from jax import random

import numpyro
import numpyro.distributions as dist
from numpyro import util
from numpyro.contrib.control_flow import scan
from numpyro.infer import SVI, TraceEnum_ELBO, init_to_median, Predictive
from numpyro.infer.autoguide import AutoNormal
from numpyro.optim import optax_to_numpyro

import optax
from numpyro import handlers
from scipy.stats import norm, f_oneway

import ast
import regex as re

from scipy.special import logsumexp
import seaborn as sns
import matplotlib.pyplot as plt
import pickle

import os
import pyarrow.parquet as pq

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler


In [4]:
path = "/content/drive/MyDrive/Bayesian Final Project/DATA"

In [5]:
def read_parquet_dotted(path, columns, nrows=None, batch_size=200_000):
    # path: location of the parquet file
    # columns: list of column names to load (to avoid reading unnecessary columns)
    # nrows: optional limit on number of rows to load
    # batch_size: how many rows to read at a time when streaming the file

    pf = pq.ParquetFile(path)
    # if row limit isn't there, we read the dataset normally
    if nrows is None:
        tbl = pf.read(columns=columns)
        return tbl.to_pandas()

    out = []
    total = 0
    for batch in pf.iter_batches(batch_size=batch_size, columns=columns):
        chunk = batch.to_pandas()
        out.append(chunk)
        total += len(chunk)
        if total >= nrows:
            break
    df = pd.concat(out, ignore_index=True)
    return df.iloc[:nrows].copy()

In [6]:
df = os.path.join(path, "instance_usage_training01_parquet.parquet")
df = read_parquet_dotted(df, None)

In [7]:
instance_events = os.path.join(path, "instance_events_training01_parquet.parquet")
instance_events_df = read_parquet_dotted(instance_events, None)

In [8]:
df.head()

,start_time,collection_id,instance_index,average_usage_cpus,average_usage_memory,maximum_usage_cpus,maximum_usage_memory,cycles_per_instruction,random_sample_usage_cpus,random_sample_usage_memory,assigned_memory,page_cache_memory,memory_accesses_per_instruction,cpu_usage_distribution,tail_cpu_usage_distribution
0,1704900000000,9941075899,8,0.009247,1.756668e-03,0.028259,1.766205e-03,2.737086,0.007271,NaN,0.000000,0.000143,0.006497,"[0.00556182861328125, 0.00688934326171875, 0.0...","[0.0131988525390625, 0.0132904052734375, 0.013..."
1,2061900000000,9941075899,8,0.006569,1.771927e-03,0.022339,1.783371e-03,3.101036,0.006004,NaN,0.000000,0.000147,0.007710,"[0.00493621826171875, 0.00568389892578125, 0.0...","[0.00732421875, 0.00743865966796875, 0.0078125..."
2,307200000000,19542402683,12,0.001299,1.531601e-03,0.012695,1.798630e-03,1.701610,0.000281,NaN,0.001953,0.000058,0.008485,"[0.00019359588623046875, 0.0002117156982421875...","[0.004913330078125, 0.0050048828125, 0.0051498..."
3,166614000000,84700535897,0,0.001465,7.057190e-05,0.003113,1.316071e-04,NaN,0.002598,NaN,0.000185,0.000070,NaN,"[3.719329833984375e-05, 3.719329833984375e-05,...","[0.002979278564453125, 0.002979278564453125, 0..."
4,346323000000,84700535897,0,0.000000,9.536743e-07,0.000000,9.536743e-07,NaN,0.000000,NaN,0.000177,0.000000,NaN,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]"


In [9]:
instance_events_df.head()

,collection_id,instance_index,time,type
0,330587125888,13,2305877675138,0
1,330587125888,406,2337139488043,0
2,330587125888,696,1902747071586,0
3,330587125888,265,1280003584881,0
4,330587125888,557,1852008055701,0


Applying the same merging and normalization as our train_df that the model was trained on

In [10]:
# merging based on last_event being one of the 2 below aka fail, kill
FAIL_TYPES = {5, 7}  # fail, kill (removed 4 aka evict)

instance_events_df["time"] = pd.to_numeric(instance_events_df["time"], errors="coerce")

# last event row per (collection_id, instance_index)
last_rows = instance_events_df.loc[
    instance_events_df.groupby(["collection_id", "instance_index"])["time"].idxmax(),
    ["collection_id", "instance_index", "type"]
].copy()

last_rows["last_event"] = last_rows["type"].isin(FAIL_TYPES).astype("int8")

# merge/broadcast onto df
df = df.merge(
    last_rows[["collection_id", "instance_index", "last_event"]],
    on=["collection_id", "instance_index"],
    how="left"
)

df["last_event"] = df["last_event"].fillna(0).astype("int8")

In [11]:
def parse_dist(val, expected_len=11):
    # Define a function that parses a CPU distribution field into numeric values.
    # val: the raw value coming from the dataset (can be list, numpy array, or string).
    # expected_len: the number of values we expect in the distribution (default = 11 percentiles).
    try:
        if isinstance(val, (list, np.ndarray)):
            vals = list(val)
        elif isinstance(val, str):
            nums = re.findall(r'[-+]?\d*\.?\d+(?:[eE][-+]?\d+)?', val)
            vals = [float(x) for x in nums]
        else:
            return [np.nan] * expected_len
        return vals[:expected_len] + [np.nan] * max(0, expected_len - len(vals))
    except:
        return [np.nan] * expected_len

In [12]:
def clean_cpu_usage_distribution(df):
    # Define a function that cleans and expands the "cpu_usage_distribution" column in a DataFrame.
    # df: pandas DataFrame containing a column called "cpu_usage_distribution"
    # which stores CPU usage percentile distributions.
    # Result:
    # dist_parsed becomes a pandas Series where each element is a list like:
    # [p0, p10, p20, ..., p100]
    dist_parsed = df["cpu_usage_distribution"].apply(parse_dist)
    pct_cols = [f"cpu_p{i}" for i in range(0, 101, 10)]

    dist_df = pd.DataFrame(dist_parsed.tolist(), columns=pct_cols)
    df[pct_cols] = dist_df
    df["cpu_burstiness"] = df["cpu_p90"] - df["cpu_p10"]
    return df

In [13]:
def extract_tail_cpu_features(df):
    # Define a function that extracts statistical features from the
    # "tail_cpu_usage_distribution" column in a DataFrame.

    # df: pandas DataFrame containing tail CPU usage distributions
    #     (usually percentiles or histogram-like values).

    # earlier, using parse_dist we converted the distribution into a numeric list
    # we now compute summary stats from those values
    def get_stats(arr):
        arr = np.array(arr)
        clean = arr[~np.isnan(arr)]
        if len(clean)==0:
            return [np.nan]*4
        return [
            np.mean(clean),
            np.max(clean),
            np.percentile(clean, 90),
            np.mean(clean>0)
        ]

    tail_parsed = df["tail_cpu_usage_distribution"].apply(
        lambda x: get_stats(parse_dist(x, expected_len=9))
    )

    df[['tail_cpu_mean',
        'tail_cpu_max',
        'tail_cpu_p90',
        'tail_cpu_nonzero_frac']] = pd.DataFrame(tail_parsed.tolist())

    return df

In [14]:
df = clean_cpu_usage_distribution(df)
df = extract_tail_cpu_features(df)

In [15]:
# copy df and initialize new column
df["kill_row"] = 0

# make sure key/time types match
df["collection_id"] = df["collection_id"].astype("int64")
df["instance_index"] = df["instance_index"].astype("int64")
df["start_time"] = df["start_time"].astype("int64")

instance_events_df = instance_events_df.copy()
instance_events_df["collection_id"] = instance_events_df["collection_id"].astype("int64")
instance_events_df["instance_index"] = instance_events_df["instance_index"].astype("int64")
instance_events_df["time"] = instance_events_df["time"].astype("int64")
instance_events_df["type"] = instance_events_df["type"].astype("int64")

# keep only kill events
kill_events = instance_events_df[
    instance_events_df["type"].isin([5, 7])
][["collection_id", "instance_index", "time"]].copy()

# preserve original row index from df
df_work = df.reset_index().rename(columns={"index": "orig_index"})

matched_indices = []

# loop through each (collection_id, instance_index) pair that has kill events
for (cid, iid), kills_g in kill_events.groupby(["collection_id", "instance_index"]):

    df_g = df_work[
        (df_work["collection_id"] == cid) &
        (df_work["instance_index"] == iid)
    ][["orig_index", "collection_id", "instance_index", "start_time"]].copy()

    # if no rows in df for this pair, skip
    if df_g.empty:
        continue

    # sort within this one pair
    df_g = df_g.sort_values("start_time").reset_index(drop=True)
    kills_g = kills_g.sort_values("time").reset_index(drop=True)

    # nearest earlier row only
    matched_g = pd.merge_asof(
        kills_g,
        df_g,
        left_on="time",
        right_on="start_time",
        direction="backward"
    )

    # keep successful matches
    idx = matched_g["orig_index"].dropna().astype(int).tolist()
    matched_indices.extend(idx)

# mark matched rows
df.loc[np.unique(matched_indices), "kill_row"] = 1

pulling norm_params.pkl file from which model was trained on

In [16]:
norm_params_path = "/content/drive/MyDrive/Bayesian Final Project/DATA/norm_params.pkl"

with open(norm_params_path, "rb") as f:
    norm_params = pickle.load(f)

In [17]:
# apply normalization TRAIN + TEST
def apply_log_norm(df, cols, norm_params):
    for col in cols:
        eps = norm_params[col]["eps"]
        mu = norm_params[col]["mean"]
        sigma = norm_params[col]["std"]

        x = df[col].astype(float)
        df[f"{col}_logged"] = np.log(x + eps)
        df[f"{col}_logged_normed"] = (df[f"{col}_logged"] - mu) / sigma

    return df

In [18]:
cols_to_normalize = [
    'average_usage_cpus',
    'average_usage_memory',
    'maximum_usage_cpus',
    'maximum_usage_memory',
    'assigned_memory',
    'page_cache_memory',
    'cpu_p50',
    'cpu_p90',
    'cpu_p100',
    'cpu_burstiness',
    'tail_cpu_mean'
]

# filling na values
df[cols_to_normalize] = df[cols_to_normalize].fillna(0.0)

In [19]:
df = apply_log_norm(df, cols_to_normalize, norm_params)

In [20]:
df.head()

,start_time,collection_id,instance_index,average_usage_cpus,average_usage_memory,maximum_usage_cpus,maximum_usage_memory,cycles_per_instruction,random_sample_usage_cpus,random_sample_usage_memory,...,cpu_p50_logged,cpu_p50_logged_normed,cpu_p90_logged,cpu_p90_logged_normed,cpu_p100_logged,cpu_p100_logged_normed,cpu_burstiness_logged,cpu_burstiness_logged_normed,tail_cpu_mean_logged,tail_cpu_mean_logged_normed
0,1704900000000,9941075899,8,0.009247,1.756668e-03,0.028259,1.766205e-03,2.737086,0.007271,NaN,...,-4.823142,1.148561,-4.333415,0.982733,-4.158877,0.880953,-5.077847,0.913099,-4.288076,0.927558
1,2061900000000,9941075899,8,0.006569,1.771927e-03,0.022339,1.783371e-03,3.101036,0.006004,NaN,...,-5.065682,1.095887,-4.928079,0.865693,-4.395785,0.836887,-6.465318,0.643363,-4.648070,0.858526
2,307200000000,19542402683,12,0.001299,1.531601e-03,0.012695,1.798630e-03,1.701610,0.000281,NaN,...,-8.189260,0.417527,-5.342535,0.784122,-4.366515,0.842332,-5.387801,0.852841,-5.140274,0.764143
3,166614000000,84700535897,0,0.001465,7.057190e-05,0.003113,1.316071e-04,NaN,0.002598,NaN,...,-7.060432,0.662679,-5.816041,0.690928,-5.816041,0.572715,-5.828603,0.767146,-5.816041,0.634560
4,346323000000,84700535897,0,0.000000,9.536743e-07,0.000000,9.536743e-07,NaN,0.000000,NaN,...,-16.118096,-1.304411,-16.118096,-1.336687,-16.118096,-1.343499,-16.118096,-1.233217,-16.118096,-1.340933


pulling model.pkl file to use model parameters for decoding

In [21]:
file_path = "/content/drive/MyDrive/Bayesian Final Project/DATA/hmm_model_march11.pkl"

with open(file_path, "rb") as f:
    model = pickle.load(f)

pi_np = model["pi"]
A_np = model["A"]
mu_np = model["mu"]
sg_np = model["sigma"]
feature_set = model["features"]
K = model["K"]
norm_params = model["norm_params"]

applying all decoding functions onto test_df

In [22]:
def log_normal_diag(x, mu, sigma):
    # x: observation sequence (T,D)
    # mu/sigma: Gaussian parameters for each time and state (K,D)
    # returns log p(x_t | z=k) as (T,K)

    # Expand observations so they broadcast across
    x = x[:, None, :]               # (T,1,D)
    mu = mu[None, :, :]             # (1,K,D)
    sg = sigma[None, :, :]          # (1,K,D)

    # Compute Gaussian log-density and sum across feature dimension D
    return (-0.5 * (((x - mu) / sg) ** 2 + 2*np.log(sg) + np.log(2*np.pi))).sum(-1)

In [23]:
def forward_probs(X, pi, A, mu, sigma):
    # X: observation sequence (T, D)
    # pi: initial state probabilities (K,)
    # A: transition matrix (K, K)
    # mu, sigma: Gaussian parameters for each state (K, D)
    # Returns gamma_filt: filtered state probabilities (T, K)

    T = X.shape[0]
    K = pi.shape[0]

    # Compute log emission probabilities log p(x_t | z_t=k)
    logB = log_normal_diag(X, mu, sigma)   # (T, K)

    # Convert initial and transition probabilities to log-space
    logpi = np.log(pi + 1e-12)             # (K,)
    logA = np.log(A + 1e-12)               # (K, K)

    # Allocate forward probability matrix
    log_alpha = np.zeros((T, K), dtype=np.float64)

    # t = 0
    log_alpha[0] = logpi + logB[0]
    log_alpha[0] -= logsumexp(log_alpha[0])   # normalize

    # t = 1, ..., T-1
    for t in range(1, T):
        # for each current state k:
        # log_alpha[t, k] = logB[t, k] + logsumexp_j(log_alpha[t-1, j] + logA[j, k])
        trans_scores = log_alpha[t - 1][:, None] + logA   # (K, K)
        log_alpha[t] = logB[t] + logsumexp(trans_scores, axis=0)
        log_alpha[t] -= logsumexp(log_alpha[t])           # normalize

    gamma_filt = np.exp(log_alpha)   # (T, K), rows sum to ~1
    return gamma_filt

In [24]:
def add_forward_state_probs(df, feature_set, pi, A, mu, sigma, time_col="start_time"):
    # df: dataframe containing job observations
    # feature_set: columns used as HMM features
    # pi, A, mu, sigma: learned HMM parameters
    # Returns dataframe with state probability columns added
    df_out = df.copy()

    K = len(pi)
    # Create column names for each hidden state probability
    prob_cols = [f"hmm_state_{k}_prob" for k in range(K)]

    # initialize probability columns
    for col in prob_cols:
        df_out[col] = np.nan

    # preserve original row identity so we can write back in correct order
    df_out["_orig_idx"] = np.arange(len(df_out))

    for (cid, iid), g in df_out.groupby(["collection_id", "instance_index"], sort=False):
        # Process each job instance in time order
        g_sorted = g.sort_values(time_col)

        # Extract feature sequence
        X = g_sorted[feature_set].to_numpy(dtype=np.float32)

        if X.shape[0] == 0:
            continue

        # Compute filtered state probabilities
        probs = forward_probs(X, pi, A, mu, sigma)   # (T, K)

        # assign back row-by-row in sorted time order
        df_out.loc[g_sorted.index, prob_cols] = probs

    df_out = df_out.drop(columns="_orig_idx")
    return df_out

In [25]:
def viterbi_path(X, pi, A, mu, sigma):
    # X: observation sequence (T, D)
    # pi: initial state probabilities (K,)
    # A: transition matrix (K, K)
    # mu, sigma: Gaussian emission parameters (K, D)
    # Returns path: most likely hidden state sequence (T,)

    T = X.shape[0]
    K = pi.shape[0]

    # Compute emission log probabilities
    logB = log_normal_diag(X, mu, sigma)   # (T, K)
    logpi = np.log(pi + 1e-12)
    logA = np.log(A + 1e-12)

    delta = np.zeros((T, K), dtype=np.float64)
    psi = np.zeros((T, K), dtype=np.int32)

    # initialization
    delta[0] = logpi + logB[0]

    # recursion
    for t in range(1, T):
        for k in range(K):
            scores = delta[t - 1] + logA[:, k]
            psi[t, k] = np.argmax(scores)
            delta[t, k] = np.max(scores) + logB[t, k]

    # backtrack
    path = np.zeros(T, dtype=np.int32)
    path[-1] = np.argmax(delta[-1])

    for t in range(T - 2, -1, -1):
        path[t] = psi[t + 1, path[t + 1]]

    return path

In [26]:
def add_viterbi_state_column(df, feature_set, pi, A, mu, sigma, time_col="start_time"):
    # df: dataframe containing job observations
    # feature_set: columns used as HMM features
    # pi, A, mu, sigma: learned HMM parameters
    # Returns dataframe with a new column "viterbi_state"

    df_out = df.copy()
    df_out["viterbi_state"] = np.nan

    for (cid, iid), g in df_out.groupby(["collection_id", "instance_index"], sort=False):
        g_sorted = g.sort_values(time_col)

        X = g_sorted[feature_set].to_numpy(dtype=np.float32)

        if len(X) == 0:
            continue

        path = viterbi_path(X, pi, A, mu, sigma)

        df_out.loc[g_sorted.index, "viterbi_state"] = path

    df_out["viterbi_state"] = df_out["viterbi_state"].astype("Int64")
    return df_out

In [27]:
df = add_forward_state_probs(
    df=df,
    feature_set=feature_set,
    pi=pi_np,
    A=A_np,
    mu=mu_np,
    sigma=sg_np,
    time_col="start_time"
)

df["dominant_forward_state"] = df[
    ["hmm_state_0_prob", "hmm_state_1_prob", "hmm_state_2_prob"]
].to_numpy().argmax(axis=1)

In [28]:
df = add_viterbi_state_column(
    df=df,
    feature_set=feature_set,
    pi=pi_np,
    A=A_np,
    mu=mu_np,
    sigma=sg_np,
    time_col="start_time"
)

In [29]:
df[["start_time", "collection_id", "instance_index", "hmm_state_0_prob", "hmm_state_1_prob", "hmm_state_2_prob", "viterbi_state"]].head()

,start_time,collection_id,instance_index,hmm_state_0_prob,hmm_state_1_prob,hmm_state_2_prob,viterbi_state
0,1704900000000,9941075899,8,1.000000,2.309993e-27,1.162457e-11,0
1,2061900000000,9941075899,8,1.000000,2.121209e-26,4.650603e-12,0
2,307200000000,19542402683,12,0.000030,2.318772e-09,9.999700e-01,2
3,166614000000,84700535897,0,0.265623,2.185215e-06,7.343750e-01,0
4,346323000000,84700535897,0,0.643197,3.568034e-01,5.815083e-29,0


Forward states and viterbi states have now successfully been predicted for test_df. now we will check $P(\text{kill} \mid \text{state})$. That is for each state, what percent of rows are a kill row?

In [30]:
# what percent of kill rows are in each state?
state_kill_rate = (
    df.groupby("dominant_forward_state")["kill_row"]
    .mean()
    .mul(100)
    .reset_index(name="kill_rate_percent")
)

print(state_kill_rate)

   dominant_forward_state  kill_rate_percent
0                       0           0.184429
1                       1           0.232824
2                       2           0.020363


We are now predicting $P(\text{state} \mid \text{kill})$. That is, if a kill happened, what state was it in?

In [31]:
# what percent of rows are a kill row for a given state?
# only rows marked as kill rows
kill_df = df[df["kill_row"] == 1].copy()

# dominant forward state breakdown
dominant_breakdown = (
    kill_df.groupby("dominant_forward_state")
    .size()
    .div(len(kill_df))
    .mul(100)
    .reset_index(name="percent")
)

print("Dominant forward state breakdown among kill_row == 1:")
print(dominant_breakdown)

# viterbi state breakdown
viterbi_breakdown = (
    kill_df.groupby("viterbi_state")
    .size()
    .div(len(kill_df))
    .mul(100)
    .reset_index(name="percent")
)

print("\nViterbi state breakdown among kill_row == 1:")
print(viterbi_breakdown)

Dominant forward state breakdown among kill_row == 1:
   dominant_forward_state    percent
0                       0  12.908821
1                       1  77.998018
2                       2   9.093162

Viterbi state breakdown among kill_row == 1:
   viterbi_state    percent
0              0  10.282458
1              1  80.773043
2              2   8.944500


### State Risk Relative to Baseline Frequency

This metric measures whether a state is **overrepresented in failures relative to how often it appears overall**.

Instead of only looking at:

- $P(\text{kill} \mid \text{state})$ → how dangerous a state is  
- $P(\text{state} \mid \text{kill})$ → where kills tend to occur  

we compare **kill frequency to baseline state frequency**.

---

### The Idea

We compute:
$
\
\frac{P(\text{state} \mid \text{kill})}{P(\text{state})}
\
$
Where:

- $P(\text{state})$ = how common the state is in the dataset overall  
- $P(\text{state} \mid \text{kill})$ = how often kills occur in that state  

---

### Interpretation

| Value | Meaning |
|------|--------|
| > 1 | State produces **more kills than expected** |
| = 1 | Neutral |
| < 1 | State produces **fewer kills than expected** |

---

### Example

| State | \(P(\text{state})\) | \(P(\text{state} \mid \text{kill})\) | Ratio |
|------|------|------|------|
| 0 | 40% | 10% | 0.25 |
| 1 | 20% | 25% | 1.25 |
| 2 | 10% | 50% | **5.0** |

State **2 is extremely risky**, even though it occurs relatively rarely.

---

### Why This Is Useful

This removes bias from **common states**.  
A state that appears frequently may naturally contain many kills simply due to volume.

This metric highlights states where **kills occur disproportionately often relative to their baseline frequency**, making it easier to identify **true failure states** in the HMM.

In [32]:
# overall state frequency
state_freq = (
    df.groupby("dominant_forward_state")
    .size()
    .div(len(df))
    .reset_index(name="p_state")
)

# kill state frequency
kill_df = df[df["kill_row"] == 1]

state_given_kill = (
    kill_df.groupby("dominant_forward_state")
    .size()
    .div(len(kill_df))
    .reset_index(name="p_state_given_kill")
)

# merge
state_risk = state_freq.merge(state_given_kill, on="dominant_forward_state")

# risk ratio
state_risk["risk_ratio"] = state_risk["p_state_given_kill"] / state_risk["p_state"]

print(state_risk.sort_values("risk_ratio", ascending=False))

   dominant_forward_state   p_state  p_state_given_kill  risk_ratio
1                       1  0.393412            0.779980    1.982603
0                       0  0.082196            0.129088    1.570498
2                       2  0.524392            0.090932    0.173404


In [34]:
# rows leading up to kills
pre_kill = df[df["kill_row"] == 1].copy()

pre_kill["prev2_state"] = df.groupby(
    ["collection_id","instance_index"]
)["dominant_forward_state"].shift(2)

pre_kill[["prev2_state","prev_state","dominant_forward_state"]].value_counts()

prev2_state  prev_state  dominant_forward_state
1.0          1.0         1                         1301
2.0          1.0         1                          587
1.0          2.0         1                          567
2.0          2.0         1                          414
0.0          0.0         0                          133
2.0          2.0         2                          121
1.0          1.0         0                          114
             0.0         1                           97
0.0          1.0         1                           88
1.0          1.0         2                           82
             2.0         0                           64
2.0          1.0         2                           61
             2.0         0                           60
1.0          2.0         2                           60
2.0          1.0         0                           59
0.0          2.0         1                           46
2.0          0.0         1                           37
                         0                           35
0.0          2.0         0                           31
             1.0         2                           13
1.0          0.0         0                           13
0.0          1.0         0                           12
             0.0         1                           10
2.0          0.0         2                            9
0.0          2.0         2                            8
1.0          0.0         2                            7
0.0          0.0         2                            3
Name: count, dtype: int64